# Notebook 8 — Final Evaluation

**The only notebook that touches the test partition, and it is gated.**

## What changed from R01

| Change | Reason |
|---|---|
| Test scoring gated on two flags **and** a clean git commit | GATE-1(iii), Q25 C6: the same seed-42 partition was scored in Notebooks 4, 5b, 6, 6b, 7 and 8 |
| Every scoring event appended to the run manifest | If you score twice, the record says so. That is the honest version |
| Fairness threshold read from `config.FAIRNESS_THRESHOLD` | Q16: R10 cited "the 0.10 threshold set in Methods" and no Method section set one |
| Subgroup positive counts reported beside every difference | Q16: the geographic-zone difference of 0.167 was reported in a closing sentence and never diagnosed |
| Geographic zone cross-tabulated against school | Q16: zone spreads across four schools, so the subgroup contrast is substantially a school contrast |
| Threshold-to-caseload table | Q12: nothing in R01 converted AUC-PR into pupils flagged per hundred |
| Clearance certificate scored **from the CSVs** | Q19, Q25: R01's Table 6 was scored partly on intent and came out 5 of 6; the examiner re-scored it at 1 of 6 |
| Timings regenerated | GATE-3: R9 said 0.402 ± 0.176 s; the committed CSV said 0.1457 ± 0.0349 s |
| No local redefinition of the focal loss | R01 redefined it here so pickle could resolve `__main__.focal_loss_lgb`. Models are now saved against the `losses` module |

## Before you run Section 3

1. Fix every config value in `config.py`.
2. Run Notebooks 1 through 7 and 9, and read them.
3. Commit everything, including the results directories.
4. Set `SCORE_TEST = True` and `FREEZE_CONFIRMED = True` in `config.py`.

Then run once.

In [ ]:
# ---- bootstrap: repo-relative imports, no drive.mount, no hard-coded path ----
import sys, os
from pathlib import Path

def _find_repo(start=None):
    p = Path(start or Path.cwd()).resolve()
    for c in [p, *p.parents]:
        if (c / "config.py").exists():
            return c
    return p

REPO = Path(os.environ["DROPOUT_REPO"]) if os.environ.get("DROPOUT_REPO") else _find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# In Colab, clone the repo first, then run:
#     import os; os.environ["DROPOUT_REPO"] = "/content/student-dropout-prediction-ghana"
# Raw pupil-level data is NOT in the repo (ethics); place it under data-raw/
# locally. Nothing below calls drive.mount().

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from config import *
from pipeline import (preprocess_inside_fold, frozen_split, cv_splits,
                      run_grid, compare_arms, two_level_variance, fit_arm,
                      score_binary, caseload_table, fixed_capacity_recall,
                      ARMS, ARM_LABELS, HEADLINE, OLD_HEADLINE, school_splits)

banner("NOTEBOOK 8 — FINAL EVALUATION")

# Capture git state BEFORE creating any output directory. Running a notebook
# writes results, which makes the tree dirty; if that counted, the freeze
# check below could never pass. git_info() excludes output paths, and
# capturing first keeps the record about the SOURCE that produced the numbers.
GIT = git_info()
OUT = run_dir("notebook08_final")
ENV = capture_environment(OUT)

df = pd.read_csv(CLEANED_CSV)
train_pool, test_holdout = frozen_split(df)

print("\n--- M19 REPLACEMENT TABLE (paste, do not retype) ---")
print(ENV.to_string(index=False))
print("\nReconcile requirements.txt and environment.yml against "
      f"{(OUT/'pip_freeze.txt').name}. All three must agree. The R01 M19 table "
      "disagreed with both on every entry.")
print(f"\ngit: {GIT}")

In [ ]:
# ---- 1. CV results on the training pool (no test data) ---------------
fold_df, preds = run_grid(train_pool, seeds=SEEDS, collect_predictions=True)
fold_df.to_csv(OUT / "final_fold_scores.csv", index=False)
var = two_level_variance(fold_df, PRIMARY_METRIC)
var.to_csv(OUT / "final_variance.csv", index=False)

CONTRASTS = [
    ("MATCHED (1 change: objective only)", *HEADLINE),
    ("AS PUBLISHED (3 changes)", *OLD_HEADLINE),
    ("focal on raw features (1 change)", "A_raw_ce_noW", "C_raw_focal_noW"),
    ("composites only (1 change)", "A_raw_ce_noW", "E_all_ce_noW"),
]
rows = []
for label, ref, test in CONTRASTS:
    _, s = compare_arms(fold_df, ref, test, label)
    rows.append(s)
contrast = pd.DataFrame(rows)
contrast.to_csv(OUT / "final_contrasts.csv", index=False)
print(contrast[["contrast", "grand_mean_diff", "between_seed_sd",
                "ci95_lo", "ci95_hi", "n_seeds_favouring_test",
                "n_seeds_p_below_alpha"]].round(5).to_string(index=False))
HEAD = contrast.iloc[0]

In [ ]:
# ---- 2. THRESHOLD TO CASELOAD (Q12) ---------------------------------
case = caseload_table(preds, HEADLINE[1])
cap = fixed_capacity_recall(preds, HEADLINE[1])
case.to_csv(OUT / "threshold_to_caseload.csv", index=False)
cap.to_csv(OUT / "fixed_capacity_recall.csv", index=False)

BASE = float(train_pool[TARGET].mean())
print(f"THRESHOLD TO CASELOAD (base rate {100*BASE:.1f}%, "
      f"{int(train_pool[TARGET].sum())} dropout cases)\n")
print(case.to_string(index=False))
print("\nA headteacher who can review only k pupils per 100:")
print(cap.round(3).to_string(index=False))

# The metric -> construct -> behaviour -> consequence table Q12 asks for
mt = pd.DataFrame([
    {"metric": "AUC-PR", "construct": "minority-class ranking quality",
     "behavioural_meaning": "triage order: which pupils a headteacher reviews first",
     "stakeholder_consequence": f"at a capacity of 10 reviews per 100 pupils, "
        f"{100*float(cap[cap['reviewed_per_100']==10]['recall_at_capacity'].iloc[0]):.0f}% "
        f"of pupils who actually left are in the list",
     "base_rate_pct": round(100*BASE, 1)},
    {"metric": "Recall (dropout class)", "construct": "detection of departing pupils",
     "behavioural_meaning": "proportion reached before they leave",
     "stakeholder_consequence": f"at the 0.5 threshold, "
        f"{float(case[case['threshold']==0.5]['recall'].iloc[0]):.3f} of departing "
        f"pupils are flagged; "
        f"{float(case[case['threshold']==0.5]['missed_dropouts_per_100'].iloc[0]):.1f} "
        f"per 100 pupils are missed",
     "base_rate_pct": round(100*BASE, 1)},
    {"metric": "Precision (dropout class)", "construct": "review burden",
     "behavioural_meaning": "share of flagged pupils who genuinely need review",
     "stakeholder_consequence": f"at the 0.5 threshold, "
        f"{float(case[case['threshold']==0.5]['flagged_per_100'].iloc[0]):.1f} pupils "
        f"per 100 flagged, "
        f"{100*float(case[case['threshold']==0.5]['precision_among_flagged'].iloc[0]):.0f}% "
        f"of them genuine",
     "base_rate_pct": round(100*BASE, 1)},
])
mt.to_csv(OUT / "metric_to_construct_table.csv", index=False)
print("\nMETRIC -> CONSTRUCT -> BEHAVIOUR -> CONSEQUENCE (the Q12 table)\n")
for _, r in mt.iterrows():
    print(f"{r['metric']}\n  construct   : {r['construct']}"
          f"\n  behaviour   : {r['behavioural_meaning']}"
          f"\n  consequence : {r['stakeholder_consequence']}\n")
print("Print 'base rate 9.2%' beside every headline number (Q12, Q25 C5).")
print("\nNOTE: the 0.19 row is the orphan threshold in models/best_threshold.txt. "
      "It was not used for any reported result — see M15's disclosure.")

In [ ]:
# ---- 3. FAIRNESS, with the protocol declared (Q16) -------------------
print(f"declared threshold: {FAIRNESS_THRESHOLD} "
      "(config.FAIRNESS_THRESHOLD — must be in the Method BEFORE this runs)\n")

tr, vl = cv_splits(train_pool, SPLIT_SEED, n_repeats=1)[0]
X_tr, y_tr, X_vl, y_vl, _ = preprocess_inside_fold(
    train_pool.iloc[tr], train_pool.iloc[vl])
_, predict, _ = fit_arm(HEADLINE[1], X_tr, y_tr, SPLIT_SEED)
p_vl = predict(X_vl)
yhat = (p_vl >= 0.5).astype(int)
src = train_pool.iloc[vl].reset_index(drop=True)
y_arr = y_vl.to_numpy()

def audit(group_col):
    if group_col not in src.columns:
        print(f"  '{group_col}' unavailable — skipped")
        return None, None
    grp = src[group_col].astype(str).to_numpy()
    stats = []
    for gv in pd.unique(grp):
        m = grp == gv
        npos, nneg = int(y_arr[m].sum()), int((1 - y_arr[m]).sum())
        stats.append({
            "subgroup_variable": group_col, "group": gv, "n": int(m.sum()),
            "n_positive": npos, "n_negative": nneg,
            "tpr": float(yhat[m & (y_arr == 1)].mean()) if npos else np.nan,
            "fpr": float(yhat[m & (y_arr == 0)].mean()) if nneg else np.nan})
    s = pd.DataFrame(stats)
    eo = float(np.nanmax(s["tpr"]) - np.nanmin(s["tpr"]))
    eodds = float(max(eo, np.nanmax(s["fpr"]) - np.nanmin(s["fpr"])))
    print(f"\n{group_col}:")
    print(s.round(4).to_string(index=False))
    print(f"  Equal Opportunity difference : {eo:.4f}")
    print(f"  Equalized Odds difference    : {eodds:.4f}  "
          f"-> {'WITHIN' if eodds <= FAIRNESS_THRESHOLD else 'EXCEEDS'} "
          f"the declared {FAIRNESS_THRESHOLD}")
    if s["n_positive"].min() < 10:
        print(f"  CAUTION: smallest subgroup holds {int(s['n_positive'].min())} "
              "positive cases. A difference on that base is not interpretable — "
              "report the count beside it.")
    return s, {"subgroup_variable": group_col,
               "equal_opportunity_diff": eo, "equalized_odds_diff": eodds,
               "declared_threshold": FAIRNESS_THRESHOLD,
               "within_threshold": bool(eodds <= FAIRNESS_THRESHOLD),
               "n_groups": len(s),
               "min_subgroup_positives": int(s["n_positive"].min())}

detail, summ = [], []
for c in (GENDER_COL, GEO_COL, SCHOOL_COL):
    d, s = audit(c)
    if d is not None:
        detail.append(d); summ.append(s)
if detail:
    pd.concat(detail).to_csv(OUT / "fairness_subgroup_detail.csv", index=False)
    pd.DataFrame(summ).to_csv(OUT / "fairness_summary.csv", index=False)

# Is the zone disparity actually a school disparity? (Q16)
if GEO_COL in src.columns and SCHOOL_COL in src.columns:
    ct = pd.crosstab(src[GEO_COL], src[SCHOOL_COL])
    ct.to_csv(OUT / "zone_by_school_crosstab.csv")
    print(f"\n{GEO_COL} x {SCHOOL_COL} cross-tabulation:")
    print(ct.to_string())
    print("\nIf zones do not separate schools cleanly, the zone disparity is "
          "substantially a school disparity across four clusters with dropout "
          "rates from 3.3% to 20.7%. Diagnose it; do not report it in a "
          "closing sentence as R01 did.")

In [ ]:
# ---- 4. leave-one-school-out (GATE-1 iv) ----------------------------
splits = school_splits(df)
if splits:
    rows = []
    for fi, (tri, vli, held) in enumerate(splits, 1):
        X_a, y_a, X_b, y_b, _ = preprocess_inside_fold(df.iloc[tri], df.iloc[vli])
        if y_b.nunique() < 2:
            print(f"  school {held}: single class held out, skipped")
            continue
        for arm in (HEADLINE[0], HEADLINE[1], OLD_HEADLINE[0]):
            _, pr, _ = fit_arm(arm, X_a, y_a, SPLIT_SEED)
            rows.append({"held_out_school": held, "arm": arm,
                         **score_binary(y_b, pr(X_b))})
    logo = pd.DataFrame(rows)
    logo.to_csv(OUT / "leave_one_school_out.csv", index=False)
    print(logo[["held_out_school", "arm", "n", "n_positive",
                "base_rate_pct", "auc_pr", "recall"]].round(4).to_string(index=False))
    print("\nmean AUC-PR per arm, school-held-out:")
    print(logo.groupby("arm")["auc_pr"].agg(["mean", "std", "count"]).round(4).to_string())
    cv_mean = fold_df[fold_df["arm"] == HEADLINE[1]]["auc_pr"].mean()
    lo_mean = logo[logo["arm"] == HEADLINE[1]]["auc_pr"].mean()
    print(f"\npupil-level CV {cv_mean:.4f}  vs  school-held-out {lo_mean:.4f}  "
          f"(drop {cv_mean-lo_mean:+.4f})")
    print("""
THIS IS THE MOST IMPORTANT NUMBER IN THE PROJECT.
A large drop means performance depended on having seen the school, and the
0.99 AUC-PR was substantially a cluster artefact. That is the paper the
examination's DAT countersign pointed at -- far more interesting than a null
on a loss function. A small drop means the pupil-level signal transfers
across these four schools, which is a real and defensible finding.
Four clusters make this a bounded robustness check, not a generalisation
estimate. Say so.""")

In [ ]:
# ---- 5. TEST-SET SCORING — GATED ------------------------------------
if not (SCORE_TEST and FREEZE_CONFIRMED):
    print("TEST SCORING IS GATED — nothing below has run.")
    print(f"  SCORE_TEST       = {SCORE_TEST}")
    print(f"  FREEZE_CONFIRMED = {FREEZE_CONFIRMED}")
    print(f"  git commit       = {GIT['commit'][:8] or 'NONE'}")
    print(f"  source clean     = {not GIT['dirty']}"
          + (f"  (uncommitted: {GIT['dirty_paths']})" if GIT["dirty"] else ""))
    print("\nSet both in config.py only after the configuration is frozen and "
          "committed. Then run this notebook once.")
    test_df = None
else:
    assert GIT["commit"], "no git commit hash — commit before scoring the test set"
    if GIT["dirty"]:
        print("WARNING: working tree dirty at scoring time. Recorded in the manifest.")

    X_tr_f, y_tr_f, X_te, y_te, meta_f = preprocess_inside_fold(
        train_pool, test_holdout)
    print(f"final fit: {len(y_tr_f)} training rows, {meta_f['n_features']} features")
    print(f"test: {len(y_te)} rows, {int(y_te.sum())} dropout "
          f"({100*float(y_te.mean()):.1f}%)")

    rows, test_probs = [], {}
    for arm in ARMS:
        _, pr, cols = fit_arm(arm, X_tr_f, y_tr_f, SPLIT_SEED)
        p = pr(X_te); test_probs[arm] = p
        rows.append({"arm": arm, "label": ARM_LABELS[arm],
                     "n_features": len(cols), "run_dir": str(OUT.name),
                     "git_commit": GIT["commit"], **score_binary(y_te, p)})
    test_df = pd.DataFrame(rows).sort_values("auc_pr", ascending=False)
    test_df.to_csv(OUT / "test_set_scored_once.csv", index=False)
    pd.DataFrame(test_probs).assign(y=y_te.to_numpy()).to_csv(
        OUT / "test_set_predictions.csv", index=False)

    print("\nTEST SET, SCORED ONCE\n")
    print(test_df[["arm", "n_features", "accuracy", "recall", "precision",
                   "auc_roc", "auc_pr", "tp", "fp", "fn", "tn"]]
          .round(4).to_string(index=False))

    # Do the arms differ at the decision level at all? (Q25 C5)
    cms = test_df.set_index("arm")[["tn", "fp", "fn", "tp"]]
    identical = cms.nunique().sum() == 4
    print(f"\nconfusion matrices identical across all arms: {identical}")
    if identical:
        print("The ranking metric moves and the decision does not. Clearance "
              "condition C5 fails on that, and it is the clearest statement of "
              "the ceiling problem: a baseline making one error on 200 cases "
              "leaves an intervention nothing to move.")

    # examine the errors (Q8)
    p_h = test_probs[HEADLINE[1]]
    fn_mask = (y_te.to_numpy() == 1) & (p_h < 0.5)
    if fn_mask.any():
        det = test_holdout.loc[X_te.index[fn_mask]].copy()
        det["predicted_probability"] = p_h[fn_mask]
        det.to_csv(OUT / "test_false_negatives.csv", index=False)
        show = [c for c in [SCHOOL_COL, GENDER_COL, GEO_COL, "average_exam_score",
                            *ATTENDANCE_COLS, SOCIOECONOMIC_COLS["family_income"],
                            "predicted_probability"] if c in det.columns]
        print(f"\nFALSE NEGATIVES ({int(fn_mask.sum())}) — examined, not just counted:")
        print(det[show].to_string())
        print("R01 reported one error as a count and never examined it. One "
              "error is a small dataset, but it is the only error you have.")

In [ ]:
# ---- 6. CLEARANCE CERTIFICATE, scored from the CSVs (Q19, Q25) ------
logo_exists = (OUT / "leave_one_school_out.csv").exists()
c1 = bool((HEAD["n_seeds_favouring_ref"] >= 9 or HEAD["n_seeds_favouring_test"] >= 9)
          and not HEAD["sd_exceeds_effect"])
c5 = (not identical) if (test_df is not None and 'identical' in dir()) else None
c6 = bool(SCORE_TEST and FREEZE_CONFIRMED and GIT["commit"] and not GIT["dirty"])

cert = pd.DataFrame([
 {"#": "C1", "condition": "Sign stable across >=10 seeds, SD smaller than the effect",
  "verdict": "PASS" if c1 else "FAIL",
  "evidence": f"{HEAD['n_seeds']} seeds; {int(HEAD['n_seeds_favouring_ref'])} favour "
              f"reference, {int(HEAD['n_seeds_favouring_test'])} favour focal; grand "
              f"mean {HEAD['grand_mean_diff']:+.4f}, between-seed SD "
              f"{HEAD['between_seed_sd']:.4f}  [final_contrasts.csv]"},
 {"#": "C2", "condition": "Survives leave-one-out (fold or cluster removed)",
  "verdict": "PASS" if logo_exists else "FAIL",
  "evidence": ("leave-one-school-out over the school clusters "
               "[leave_one_school_out.csv]; leave-one-seed-out in Notebook 9. "
               "Single-source design still precludes dataset-level LOO."
               if logo_exists else "no cluster LOO available")},
 {"#": "C3", "condition": "MDE < claimed effect, OR worded inconclusive",
  "verdict": "PASS",
  "evidence": "route two: the result is worded inconclusive throughout, never "
              "as 'no effect'. R01 scored this FAIL against itself; it passes."},
 {"#": "C4", "condition": "Baseline received the same tuning budget, and it is stated",
  "verdict": "PASS",
  "evidence": f"{HEADLINE[0]} and {HEADLINE[1]} take the identical feature "
              f"matrix and the identical weighting mechanism; zero search "
              f"trials each; full search history disclosed in Notebook 6 "
              f"[search_disclosure_for_M13.csv]"},
 {"#": "C5", "condition": "Metric moves with the claimed decision; base rate printed",
  "verdict": "PASS" if c5 else ("FAIL" if c5 is False else "PENDING TEST RUN"),
  "evidence": f"threshold_to_caseload.csv and metric_to_construct_table.csv "
              f"carry the metric through to a caseload; base rate "
              f"{100*BASE:.1f}% printed beside every figure. "
              f"{'Confusion matrices identical across arms.' if c5 is False else ''}"},
 {"#": "C6", "condition": "Test set scored once, after freeze, committed",
  "verdict": "PASS" if c6 else "FAIL",
  "evidence": f"run {OUT.name}, commit {GIT['commit'][:8] or 'NONE'}, "
              f"dirty={GIT['dirty']}. NOTE: for the R01 RESULT this fails "
              f"permanently — that test partition was scored across six "
              f"notebooks. State it in the limitations rather than attempting "
              f"a reconstruction."},
])
cert.to_csv(OUT / "clearance_certificate.csv", index=False)

print("CLEARANCE CERTIFICATE — every cell cites a number or a hash")
print("=" * 72)
for _, r in cert.iterrows():
    print(f"{r['#']}  {r['verdict']:<14s} {r['condition']}")
    print(f"       {r['evidence']}\n")
n_pass = int((cert["verdict"] == "PASS").sum())
print(f"SCORE: {n_pass} of 6")
print("\nReplace Table 6 wholesale. Do not edit individual cells — the row "
      "that moves in your favour (C3) matters as much as the ones that move "
      "against.")

In [ ]:
# ---- 7. timings and manifest ----------------------------------------
timing = (fold_df.groupby("arm")["fit_seconds"]
          .agg(mean_fit_seconds="mean", sd_fit_seconds="std", n_fits="size")
          .reset_index())
timing["run_dir"] = OUT.name
timing.to_csv(OUT / "training_times.csv", index=False)
print("R9 REPLACEMENT — regenerated in the environment captured above\n")
print(timing.round(4).to_string(index=False))
a, b = HEADLINE
ma = float(timing.set_index("arm").loc[a, "mean_fit_seconds"])
mb = float(timing.set_index("arm").loc[b, "mean_fit_seconds"])
print(f"\n{b} vs {a}: {100*(mb-ma)/ma:+.1f}% mean fit time")
print("R01 reported 0.402 +/- 0.176 s against a committed CSV of "
      "0.1457 +/- 0.0349 s. Report what this prints.")
print("\nKeep R9's existing caveat verbatim — that the difference cannot be "
      "attributed to the focal computation from these measurements alone. "
      "The examination singled it out as the register the rest of the "
      "manuscript needs.")

extra = {"notebook": "08_final", "git": GIT,
         "test_scored": bool(test_df is not None),
         "clearance_score": f"{n_pass}/6",
         "fairness_threshold_declared": FAIRNESS_THRESHOLD,
         "headline_contrast": HEAD["contrast"],
         "headline_grand_mean_diff": float(HEAD["grand_mean_diff"]),
         "headline_between_seed_sd": float(HEAD["between_seed_sd"])}
if test_df is not None:
    extra["test_scoring_events"] = [{
        "utc": pd.Timestamp.utcnow().isoformat(),
        "git_commit": GIT["commit"], "dirty": GIT["dirty"],
        "n_arms": len(ARMS)}]
write_manifest(OUT, extra)

files = sorted(p.relative_to(OUT).as_posix() for p in OUT.rglob("*") if p.is_file())
(OUT / "INDEX.txt").write_text("\n".join(files))
print(f"\n{len(files)} files written to {OUT}")
print(f"Cite run '{OUT.name}' in M14, M19, M21 and every table caption, so "
      "every reported number traces to one locked run (GATE-3).")